# Hartwell University module registry: synthetic data generator

This notebook produces `hartwell_registry.db`, a SQLite database for an imaginary university faculty.
It covers departments, academic staff, degree programmes, students, modules, the yearly runs of each
module, enrolments, assessment components and individual submissions with marks, for the academic
years 2023/24 to 2025/26.

All values are generated here (Faker for names, NumPy for everything numeric). The seed is fixed, so the
output is reproducible. The schema is defined near the end, just before loading, once the generated
records are in their final shape.

In [1]:
try:
    import faker
except ImportError:
    %pip install -q faker

In [2]:
import os
import sqlite3
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker

SEED = 8861
gen = np.random.default_rng(SEED)
names = Faker('en_GB')
names.seed_instance(SEED)
names_intl = Faker(['zh_CN', 'hi_IN', 'en_US', 'es_ES', 'ar_AA', 'pl_PL'])
names_intl.seed_instance(SEED + 7)

OUT = 'hartwell_registry.db'
YEARS = ['2023/24', '2024/25', '2025/26']          # academic years covered
YEAR_START = {y: int(y[:4]) for y in YEARS}

def pick(options, weights=None):
    """Weighted choice returning a plain Python value."""
    i = gen.choice(len(options), p=None if weights is None else np.array(weights) / sum(weights))
    return options[i]

## Departments, staff and programmes

Five departments sit in the faculty. Staff grade drives salary-related fields in a real HR system, so here
it only drives seniority patterns: professors are fewer and more likely to lead level 6 and 7 modules.

In [3]:
DEPTS = [
    # code, name, building, established
    ('COM', 'Computer Science', 'Turing Building', 1987),
    ('ENG', 'Engineering', 'Brunel Building', 1962),
    ('MAT', 'Mathematics', 'Noether Building', 1962),
    ('PHY', 'Physics and Astronomy', 'Faraday Building', 1965),
    ('DSC', 'Data Science and Statistics', 'Turing Building', 2016),
]
departments = [dict(dept_code=c, dept_name=n, building=b, established_year=y) for c, n, b, y in DEPTS]

GRADES = ['Lecturer', 'Senior Lecturer', 'Associate Professor', 'Professor']
staff = []
used_mail = set()
for d in departments:
    headcount = int(gen.integers(11, 17))
    for _ in range(headcount):
        grade = pick(GRADES, [40, 32, 16, 12])
        first, last = names.first_name(), names.last_name()
        handle = (first[0] + '.' + ''.join(ch for ch in last if ch.isalpha())).lower()
        mail = handle + '@hartwell.ac.uk'
        k = 2
        while mail in used_mail:
            mail = handle + str(k) + '@hartwell.ac.uk'; k += 1
        used_mail.add(mail)
        staff.append(dict(staff_id=len(staff) + 1001, dept_code=d['dept_code'],
                          title='Prof' if grade == 'Professor' else pick(['Dr', 'Dr', 'Dr', 'Mr', 'Ms']),
                          first_name=first, last_name=last, grade=grade,
                          fte=float(pick([1.0, 1.0, 1.0, 0.8, 0.6, 0.5])), email=mail))

PROGRAMMES = [
    # code, title, dept, award, years, mode
    ('BSC-CS', 'Computer Science', 'COM', 'BSc', 3, 'Full-time'),
    ('BSC-SE', 'Software Engineering', 'COM', 'BSc', 4, 'Sandwich'),
    ('BSC-CYB', 'Cyber Security', 'COM', 'BSc', 3, 'Full-time'),
    ('MSC-AI', 'Artificial Intelligence', 'COM', 'MSc', 1, 'Full-time'),
    ('BENG-ME', 'Mechanical Engineering', 'ENG', 'BEng', 3, 'Full-time'),
    ('MENG-AE', 'Aerospace Engineering', 'ENG', 'MEng', 4, 'Full-time'),
    ('BSC-MA', 'Mathematics', 'MAT', 'BSc', 3, 'Full-time'),
    ('BSC-MAP', 'Mathematics', 'MAT', 'BSc', 6, 'Part-time'),
    ('BSC-PH', 'Physics', 'PHY', 'BSc', 3, 'Full-time'),
    ('MPHYS', 'Physics with Astrophysics', 'PHY', 'MEng', 4, 'Full-time'),
    ('BSC-DS', 'Data Science', 'DSC', 'BSc', 3, 'Full-time'),
    ('MSC-DS', 'Data Science', 'DSC', 'MSc', 1, 'Full-time'),
]
programmes = [dict(programme_code=c, title=t, dept_code=d, award=a, duration_years=y, mode=m)
              for c, t, d, a, y, m in PROGRAMMES]
print(len(staff), 'staff,', len(programmes), 'programmes')

67 staff, 12 programmes


## Modules and their yearly runs

Each department offers four modules per level. Codes follow the common UK pattern of level digit,
subject letters and a serial number. Level 4 to 6 modules are 30 credits; level 7 modules are 15 or 30.
A module "run" is one delivery of a module in one academic year and semester.

In [4]:
TOPICS = {
    'COM': ['Programming', 'Computer Systems', 'Databases', 'Web Technologies', 'Algorithms', 'Networks',
            'Software Engineering', 'Operating Systems', 'Machine Learning', 'Security', 'Distributed Systems',
            'Human Computer Interaction', 'Deep Learning', 'Cloud Computing', 'Research Methods', 'Computer Vision'],
    'ENG': ['Engineering Mathematics', 'Statics', 'Materials', 'Design Practice', 'Thermodynamics', 'Fluid Mechanics',
            'Control Systems', 'Dynamics', 'Finite Element Analysis', 'Propulsion', 'Structures', 'Manufacturing',
            'Aerodynamics', 'Systems Engineering', 'Composites', 'Project Management'],
    'MAT': ['Calculus', 'Linear Algebra', 'Probability', 'Proof and Logic', 'Real Analysis', 'Differential Equations',
            'Group Theory', 'Numerical Methods', 'Complex Analysis', 'Topology', 'Optimisation', 'Number Theory',
            'Functional Analysis', 'Stochastic Processes', 'Measure Theory', 'Mathematical Modelling'],
    'PHY': ['Mechanics', 'Waves and Optics', 'Laboratory Physics', 'Mathematical Physics', 'Electromagnetism',
            'Quantum Physics', 'Thermal Physics', 'Astronomy', 'Solid State Physics', 'Nuclear Physics', 'Cosmology',
            'Computational Physics', 'Particle Physics', 'Stellar Astrophysics', 'General Relativity', 'Photonics'],
    'DSC': ['Statistics', 'Data Handling', 'Python for Data', 'Data Visualisation', 'Regression', 'Data Mining',
            'Big Data', 'Bayesian Statistics', 'Time Series', 'Data Ethics', 'Statistical Learning', 'Experimental Design',
            'Advanced Data Mining', 'Text Analytics', 'Data Science Project', 'Neural Networks'],
}
modules = []
for d in departments:
    topics = TOPICS[d['dept_code']]
    dept_staff = [s for s in staff if s['dept_code'] == d['dept_code']]
    for li, level in enumerate([4, 5, 6, 7]):
        for k in range(4):
            senior = [s for s in dept_staff if s['grade'] in ('Associate Professor', 'Professor', 'Senior Lecturer')]
            pool = senior if level >= 6 and senior and gen.random() < 0.7 else dept_staff
            leader = pool[int(gen.integers(0, len(pool)))]
            modules.append(dict(module_code='%d%s%04d' % (level, d['dept_code'], 1001 + li * 10 + k),
                                title=topics[li * 4 + k], dept_code=d['dept_code'], level=level,
                                credits=30 if level < 7 else pick([15, 30]), leader_id=leader['staff_id']))

module_runs = []
for m in modules:
    sem = pick(['A', 'B', 'AB'], [35, 35, 30])
    for y in YEARS:
        leader = m['leader_id'] if gen.random() < 0.85 else pick([s['staff_id'] for s in staff if s['dept_code'] == m['dept_code']])
        module_runs.append(dict(module_code=m['module_code'], academic_year=y, semester=sem,
                                leader_id=leader, capacity=int(pick([60, 80, 120, 150, 200]))))
print(len(modules), 'modules,', len(module_runs), 'runs')

80 modules, 240 runs


## Students

Students enter between 2019 and 2025. Entry year and programme length decide which levels they study
inside the three-year window. Sandwich students spend their third year on placement and part-time
students take half the credits each year.

In [5]:
students = []
dept_of = {p['programme_code']: p for p in programmes}
prog_weights = [190, 70, 90, 110, 120, 70, 110, 25, 80, 50, 120, 90]
for _ in range(1650):
    prog = pick(programmes, prog_weights)
    pg = prog['award'] == 'MSc'
    entry = int(gen.integers(2023, 2026)) if pg else int(gen.integers(2019 if prog['mode'] == 'Part-time' else 2020, 2026))
    intl = gen.random() < (0.55 if pg else 0.18)
    src = names_intl if intl else names
    first, last = src.first_name(), src.last_name()
    age_at_entry = int(np.clip(gen.normal(23, 2.5), 21, 40)) if pg else int(np.clip(gen.gamma(2, 0.7) + 18, 18, 45))
    if prog['mode'] == 'Part-time': age_at_entry += int(gen.integers(5, 20))
    sid = int('%02d%06d' % (entry % 100, len(students) + 100000 + int(gen.integers(0, 9) * 100000)))
    clean = ''.join(ch for ch in (first + last).lower() if ch.isascii() and ch.isalpha()) or 'student'
    students.append(dict(student_id=sid, first_name=first, last_name=last,
                         uni_email='%s@student.hartwell.ac.uk' % sid,
                         personal_email='%s%d@%s' % (clean[:14], gen.integers(1, 99), pick(['gmail.com', 'outlook.com', 'hotmail.com', 'yahoo.com', 'icloud.com'], [50, 20, 12, 8, 10])),
                         programme_code=prog['programme_code'], entry_year=entry, birth_year=entry - age_at_entry,
                         fee_status='International' if intl else 'Home',
                         ability=float(np.clip(gen.normal(59, 10), 25, 90))))   # hidden, not stored
ids = [s['student_id'] for s in students]
assert len(set(ids)) == len(ids)
print(len(students), 'students')
pd.Series([s['programme_code'] for s in students]).value_counts().head()

1650 students


BSC-CS     315
BSC-DS     173
BSC-MA     162
BENG-ME    156
MSC-AI     148
Name: count, dtype: int64

## Enrolments, assessments and submissions

Every module run has two or three assessment components with weights adding up to 100. Marks come from
the student's hidden ability, a difficulty for each module and some noise per component. Coursework
tends to score a little higher than exams. Late work (up to five days) is capped at the pass mark,
which is the usual UK rule. Around 4% of students withdraw part way through a year.

In [6]:
run_key = {(r['module_code'], r['academic_year']): r for r in module_runs}
difficulty = {m['module_code']: float(gen.normal(0, 4)) for m in modules}
mods_by = {}
for m in modules:
    mods_by.setdefault((m['dept_code'], m['level']), []).append(m)

assessments = []
for r in module_runs:
    n = pick([2, 3], [60, 40])
    weights = [60, 40] if n == 2 else [40, 30, 30]
    kinds = ['Exam', 'Coursework'] if n == 2 else ['Exam', 'Coursework', pick(['Lab report', 'Presentation', 'Portfolio'])]
    if gen.random() < 0.25: kinds[0] = 'Coursework'
    y0 = YEAR_START[r['academic_year']]
    for c in range(n):
        if r['semester'] == 'A': due = date(y0, 12, 1) + timedelta(days=int(gen.integers(0, 50)))
        elif r['semester'] == 'B': due = date(y0 + 1, 4, 1) + timedelta(days=int(gen.integers(0, 60)))
        else: due = date(y0 + 1, 1, 10) + timedelta(days=int(gen.integers(0, 130)))
        assessments.append(dict(assessment_id=len(assessments) + 1, module_code=r['module_code'],
                                academic_year=r['academic_year'], semester=r['semester'], component_no=c + 1,
                                kind=kinds[c], weight_pct=weights[c], due_date=due.isoformat()))
comp_of = {}
for a in assessments:
    comp_of.setdefault((a['module_code'], a['academic_year']), []).append(a)

enrolments, submissions = [], []
for s in students:
    prog = dept_of[s['programme_code']]
    for y in YEARS:
        offset = YEAR_START[y] - s['entry_year']
        if offset < 0 or offset >= prog['duration_years']: continue
        if prog['award'] == 'MSc': level = 7
        elif prog['mode'] == 'Part-time': level = 4 + offset // 2
        elif prog['mode'] == 'Sandwich':
            if offset == 2: continue          # placement year
            level = 4 + min(offset, 3) - (1 if offset == 3 else 0)
        else: level = 4 + offset
        level = min(level, 7)
        options = mods_by[(prog['dept_code'], level)]
        take = options if prog['mode'] != 'Part-time' else options[:2] if offset % 2 == 0 else options[2:]
        withdraws = gen.random() < 0.04
        for i, m in enumerate(take):
            run = run_key[(m['module_code'], y)]
            status = 'Withdrawn' if withdraws and i >= len(take) // 2 else 'Registered'
            total = 0.0
            for a in comp_of[(m['module_code'], y)]:
                if status == 'Withdrawn' or gen.random() < 0.025:
                    continue                    # withdrawn, or simply never submitted
                lift = 4 if a['kind'] != 'Exam' else -2
                mark = s['ability'] + lift - difficulty[m['module_code']] + gen.normal(0, 8)
                late = 0
                due = datetime.fromisoformat(a['due_date'] + ' 14:00')
                if a['kind'] in ('Exam', 'Presentation'):
                    when = due + timedelta(hours=-int(gen.integers(0, 3)))
                else:
                    late = int(gen.choice([0, 1, 2, 3, 4, 5, 7], p=[.92, .03, .018, .012, .007, .005, .008]))
                    when = due - timedelta(hours=float(gen.exponential(20))) + timedelta(days=late)
                mark = float(np.clip(round(mark), 0, 100))
                if 0 < late <= 5: mark = min(mark, 40.0 if level < 7 else 50.0)
                words = None
                if a['kind'] in ('Coursework', 'Portfolio', 'Lab report'):
                    words = int(np.clip(gen.normal(2000 + 15 * a['weight_pct'] + 8 * (mark - 55), 350), 300, 6000))
                submissions.append(dict(submission_id=len(submissions) + 1, student_id=s['student_id'],
                                        assessment_id=a['assessment_id'], submitted_at=when.strftime('%Y-%m-%d %H:%M:%S'),
                                        late_days=late, mark=mark, word_count=words))
                total += mark * a['weight_pct'] / 100
            if status != 'Withdrawn':
                status = 'Passed' if total >= (50 if level == 7 else 40) else 'Failed'
            enrolments.append(dict(student_id=s['student_id'], module_code=m['module_code'], academic_year=y,
                                   semester=run['semester'], status=status))
print(len(enrolments), 'enrolments,', len(assessments), 'assessments,', len(submissions), 'submissions')

9630

 enrolments, 578 assessments, 22347 submissions


## Deliberate data quality problems

The registry is meant to look like a real one, so a measured amount of untidiness is introduced here.
Each step records how many rows it touched.

| Problem | Location | Rate | Real cause |
|---|---|---|---|
| Missing | students.personal_email | 4% | not supplied on the application |
| Missing | students.mobile | 7% | not supplied |
| Missing | submissions.mark | 1.5% | awaiting moderation when the data was extracted |
| Missing | submissions.word_count | 3% of word-counted work | similarity tool did not report a count |
| Duplicate | students | 1.5% | applicant registered twice through two routes |
| Duplicate | submissions | 0.7% | file uploaded twice by the student |

In [7]:
for s in students:
    s['mobile'] = '07%03d %06d' % (gen.integers(300, 1000), gen.integers(0, 1000000))

report = {}
def blank_out(records, field, share, tag, eligible=None):
    idx = [i for i, r in enumerate(records) if eligible is None or eligible(r)]
    hit = gen.choice(idx, size=round(len(idx) * share), replace=False)
    for i in hit:
        records[i][field] = None
    report[tag] = len(hit)

blank_out(students, 'personal_email', 0.04, 'missing personal email')
blank_out(students, 'mobile', 0.07, 'missing mobile')
blank_out(submissions, 'mark', 0.015, 'missing mark')
blank_out(submissions, 'word_count', 0.03, 'missing word count', eligible=lambda r: r['word_count'] is not None)

# second registrations: new ID, small typing differences
twice = gen.choice(len(students), size=round(len(students) * 0.015), replace=False)
copies = []
for n, i in enumerate(twice):
    c = dict(students[i])
    c['student_id'] = int('%02d%06d' % (c['entry_year'] % 100, 990000 + n))   # IDs from the late-application range
    c['uni_email'] = '%s@student.hartwell.ac.uk' % c['student_id']
    if n % 2 == 0: c['last_name'] = c['last_name'].upper()
    else: c['first_name'] = c['first_name'] + ' '
    if c['personal_email'] and n % 3 == 0: c['personal_email'] = c['personal_email'].capitalize()
    copies.append(c)
students.extend(copies)
report['duplicate students'] = len(copies)

# double uploads: same student, assessment and timestamp
again = gen.choice(len(submissions), size=round(len(submissions) * 0.007), replace=False)
for i in again:
    d = dict(submissions[i]); d['submission_id'] = len(submissions) + 1
    submissions.append(d)
report['duplicate submissions'] = len(again)

for k, v in report.items():
    print('%-26s %d' % (k, v))
assert len({s['student_id'] for s in students}) == len(students)

missing personal email     66
missing mobile             116
missing mark               335
missing word count         428
duplicate students         25
duplicate submissions      156


## Schema and load

The schema is created now and the records are inserted with foreign keys switched on. `module_runs` uses
a three-column primary key, and both `enrolments` and `assessments` point at it with a composite foreign
key, so an enrolment can only exist for a module that actually ran in that year and semester.

In [8]:
SCHEMA = """
PRAGMA foreign_keys = ON;
CREATE TABLE departments (
    dept_code        TEXT PRIMARY KEY CHECK (dept_code GLOB '[A-Z][A-Z][A-Z]'),
    dept_name        TEXT NOT NULL UNIQUE,
    building         TEXT NOT NULL,
    established_year INTEGER NOT NULL CHECK (established_year BETWEEN 1900 AND 2026)
);
CREATE TABLE staff (
    staff_id    INTEGER PRIMARY KEY CHECK (staff_id >= 1000),
    dept_code   TEXT NOT NULL REFERENCES departments (dept_code) ON UPDATE CASCADE ON DELETE RESTRICT,
    title       TEXT NOT NULL CHECK (title IN ('Prof', 'Dr', 'Mr', 'Ms', 'Mrs')),
    first_name  TEXT NOT NULL,
    last_name   TEXT NOT NULL,
    grade       TEXT NOT NULL CHECK (grade IN ('Lecturer', 'Senior Lecturer', 'Associate Professor', 'Professor')),
    fte         REAL NOT NULL CHECK (fte > 0 AND fte <= 1),
    email       TEXT NOT NULL UNIQUE CHECK (email LIKE '%@hartwell.ac.uk')
);
CREATE TABLE programmes (
    programme_code  TEXT PRIMARY KEY,
    title           TEXT NOT NULL,
    dept_code       TEXT NOT NULL REFERENCES departments (dept_code) ON UPDATE CASCADE ON DELETE RESTRICT,
    award           TEXT NOT NULL CHECK (award IN ('BSc', 'BEng', 'MEng', 'MSc')),
    duration_years  INTEGER NOT NULL CHECK (duration_years BETWEEN 1 AND 6),
    mode            TEXT NOT NULL CHECK (mode IN ('Full-time', 'Part-time', 'Sandwich')),
    UNIQUE (title, award, mode)
);
CREATE TABLE students (
    student_id      INTEGER PRIMARY KEY CHECK (student_id BETWEEN 10000000 AND 99999999),
    first_name      TEXT NOT NULL,
    last_name       TEXT NOT NULL,
    uni_email       TEXT NOT NULL UNIQUE CHECK (uni_email LIKE '%@student.hartwell.ac.uk'),
    personal_email  TEXT CHECK (personal_email IS NULL OR personal_email LIKE '%_@_%._%'),
    mobile          TEXT CHECK (mobile IS NULL OR (length(mobile) = 12 AND substr(mobile, 1, 2) = '07' AND substr(mobile, 6, 1) = ' ')),
    programme_code  TEXT NOT NULL REFERENCES programmes (programme_code) ON UPDATE CASCADE ON DELETE RESTRICT,
    entry_year      INTEGER NOT NULL CHECK (entry_year BETWEEN 2015 AND 2026),
    birth_year      INTEGER NOT NULL CHECK (birth_year BETWEEN 1950 AND 2009),
    fee_status      TEXT NOT NULL CHECK (fee_status IN ('Home', 'International'))
);
CREATE TABLE modules (
    module_code  TEXT PRIMARY KEY CHECK (module_code GLOB '[4-7][A-Z][A-Z][A-Z][0-9][0-9][0-9][0-9]'),
    title        TEXT NOT NULL,
    dept_code    TEXT NOT NULL REFERENCES departments (dept_code) ON UPDATE CASCADE ON DELETE RESTRICT,
    level        INTEGER NOT NULL CHECK (level BETWEEN 4 AND 7),
    credits      INTEGER NOT NULL CHECK (credits IN (15, 30, 60)),
    leader_id    INTEGER NOT NULL REFERENCES staff (staff_id) ON DELETE RESTRICT
);
CREATE TABLE module_runs (
    module_code    TEXT NOT NULL REFERENCES modules (module_code) ON UPDATE CASCADE ON DELETE CASCADE,
    academic_year  TEXT NOT NULL CHECK (academic_year GLOB '20[0-9][0-9]/[0-9][0-9]'),
    semester       TEXT NOT NULL CHECK (semester IN ('A', 'B', 'AB')),
    leader_id      INTEGER NOT NULL REFERENCES staff (staff_id) ON DELETE RESTRICT,
    capacity       INTEGER NOT NULL CHECK (capacity BETWEEN 10 AND 400),
    PRIMARY KEY (module_code, academic_year, semester)
);
CREATE TABLE enrolments (
    student_id     INTEGER NOT NULL REFERENCES students (student_id) ON DELETE CASCADE,
    module_code    TEXT NOT NULL,
    academic_year  TEXT NOT NULL,
    semester       TEXT NOT NULL,
    status         TEXT NOT NULL DEFAULT 'Registered' CHECK (status IN ('Registered', 'Passed', 'Failed', 'Withdrawn')),
    PRIMARY KEY (student_id, module_code, academic_year),
    FOREIGN KEY (module_code, academic_year, semester)
        REFERENCES module_runs (module_code, academic_year, semester) ON DELETE CASCADE
);
CREATE TABLE assessments (
    assessment_id  INTEGER PRIMARY KEY,
    module_code    TEXT NOT NULL,
    academic_year  TEXT NOT NULL,
    semester       TEXT NOT NULL,
    component_no   INTEGER NOT NULL CHECK (component_no BETWEEN 1 AND 5),
    kind           TEXT NOT NULL CHECK (kind IN ('Exam', 'Coursework', 'Lab report', 'Presentation', 'Portfolio')),
    weight_pct     INTEGER NOT NULL CHECK (weight_pct BETWEEN 5 AND 100),
    due_date       TEXT NOT NULL CHECK (date(due_date) = due_date),
    UNIQUE (module_code, academic_year, component_no),
    FOREIGN KEY (module_code, academic_year, semester)
        REFERENCES module_runs (module_code, academic_year, semester) ON DELETE CASCADE
);
CREATE TABLE submissions (
    submission_id  INTEGER PRIMARY KEY,
    student_id     INTEGER NOT NULL REFERENCES students (student_id) ON DELETE CASCADE,
    assessment_id  INTEGER NOT NULL REFERENCES assessments (assessment_id) ON DELETE CASCADE,
    submitted_at   TEXT NOT NULL CHECK (datetime(submitted_at) = submitted_at),
    late_days      INTEGER NOT NULL DEFAULT 0 CHECK (late_days BETWEEN 0 AND 30),
    mark           REAL CHECK (mark IS NULL OR mark BETWEEN 0 AND 100),
    word_count     INTEGER CHECK (word_count IS NULL OR word_count BETWEEN 50 AND 20000)
);
CREATE INDEX idx_sub_student ON submissions (student_id);
CREATE INDEX idx_sub_assessment ON submissions (assessment_id);
"""

def bulk_insert(conn, table, records, fields):
    marks = ', '.join('?' for _ in fields)
    conn.executemany('INSERT INTO %s (%s) VALUES (%s)' % (table, ', '.join(fields), marks),
                     [tuple(r[f] for f in fields) for r in records])

if os.path.exists(OUT):
    os.remove(OUT)
conn = sqlite3.connect(OUT)
conn.executescript(SCHEMA)
conn.execute('PRAGMA foreign_keys = ON')
bulk_insert(conn, 'departments', departments, ['dept_code', 'dept_name', 'building', 'established_year'])
bulk_insert(conn, 'staff', staff, ['staff_id', 'dept_code', 'title', 'first_name', 'last_name', 'grade', 'fte', 'email'])
bulk_insert(conn, 'programmes', programmes, ['programme_code', 'title', 'dept_code', 'award', 'duration_years', 'mode'])
bulk_insert(conn, 'students', students, ['student_id', 'first_name', 'last_name', 'uni_email', 'personal_email', 'mobile',
                                         'programme_code', 'entry_year', 'birth_year', 'fee_status'])
bulk_insert(conn, 'modules', modules, ['module_code', 'title', 'dept_code', 'level', 'credits', 'leader_id'])
bulk_insert(conn, 'module_runs', module_runs, ['module_code', 'academic_year', 'semester', 'leader_id', 'capacity'])
bulk_insert(conn, 'enrolments', enrolments, ['student_id', 'module_code', 'academic_year', 'semester', 'status'])
bulk_insert(conn, 'assessments', assessments, ['assessment_id', 'module_code', 'academic_year', 'semester',
                                               'component_no', 'kind', 'weight_pct', 'due_date'])
bulk_insert(conn, 'submissions', submissions, ['submission_id', 'student_id', 'assessment_id', 'submitted_at',
                                               'late_days', 'mark', 'word_count'])
conn.commit()
for (t,) in conn.execute("SELECT name FROM sqlite_master WHERE type = 'table'"):
    print('%-12s %6d' % (t, conn.execute('SELECT COUNT(*) FROM ' + t).fetchone()[0]))

departments       5
staff            67
programmes       12
students       1675
modules          80
module_runs     240
enrolments     9630
assessments     578
submissions   22503


## Validation

In [9]:
print('integrity:', conn.execute('PRAGMA integrity_check').fetchone()[0])
print('foreign key violations:', len(conn.execute('PRAGMA foreign_key_check').fetchall()))
look = lambda q: pd.read_sql_query(q, conn)
look("""
SELECT
 (SELECT COUNT(*) FROM students WHERE personal_email IS NULL) AS no_personal_email,
 (SELECT COUNT(*) FROM students WHERE mobile IS NULL) AS no_mobile,
 (SELECT COUNT(*) FROM submissions WHERE mark IS NULL) AS no_mark,
 (SELECT COUNT(*) FROM submissions s JOIN assessments a USING (assessment_id)
   WHERE a.kind <> 'Exam' AND a.kind <> 'Presentation' AND s.word_count IS NULL) AS no_word_count
""")

integrity: ok
foreign key violations: 0


,no_personal_email,no_mobile,no_mark,no_word_count
0,67,119,338,432


In [10]:
# the duplicates should be found again by ordinary SQL
look("""
SELECT
 (SELECT COUNT(*) - COUNT(DISTINCT lower(trim(first_name)) || '|' || lower(trim(last_name)) || '|' || birth_year
                    || '|' || IFNULL(lower(personal_email), '')) FROM students) AS repeated_students,
 (SELECT COUNT(*) FROM submissions)
   - (SELECT COUNT(*) FROM (SELECT DISTINCT student_id, assessment_id, submitted_at FROM submissions)) AS repeated_uploads
""")

,repeated_students,repeated_uploads
0,25,156


## Example questions

In [11]:
look("""
SELECT m.level, d.dept_name, ROUND(AVG(s.mark), 1) AS mean_mark, COUNT(*) AS marks
FROM submissions s JOIN assessments a USING (assessment_id)
JOIN modules m USING (module_code) JOIN departments d USING (dept_code)
WHERE s.mark IS NOT NULL
GROUP BY m.level, d.dept_name ORDER BY m.level, mean_mark DESC
""").head(10)

,level,dept_name,mean_mark,marks
0,4,Mathematics,62.3,810
1,4,Engineering,61.0,1274
2,4,Computer Science,59.7,2490
3,4,Data Science and Statistics,59.3,795
4,4,Physics and Astronomy,56.3,792
5,5,Physics and Astronomy,63.3,1016
6,5,Engineering,61.1,1228
7,5,Data Science and Statistics,59.4,852
8,5,Computer Science,59.3,2439
9,5,Mathematics,57.0,821


In [12]:
look("""
SELECT kind, ROUND(AVG(mark), 1) AS mean_mark, ROUND(AVG(late_days > 0) * 100, 1) AS pct_late
FROM submissions JOIN assessments USING (assessment_id) GROUP BY kind
""")

,kind,mean_mark,pct_late
0,Coursework,61.6,7.7
1,Exam,57.4,0.0
2,Lab report,63.3,7.3
3,Portfolio,61.2,6.4
4,Presentation,61.7,0.0


In [13]:
conn.close()
print('done:', OUT)

done: hartwell_registry.db
